# Train-time Scaling and Reinforcement Learning

> The first five lectures spent compute at **inference time**: the trained weights stay fixed, and repeated sampling, voting, reranking, and search recover correct solutions that already exist in the model.
>
> This lecture switches dimension and spends compute at **training time**, changing the parameters so the model itself becomes better at reasoning. Three papers form one line: STaR bootstraps reasoning with reasoning, DeepSeekMath scales up reinforcement learning, and DAPO fills in the engineering details of large-scale training.

Take an addition problem 23+47; the correct answer is 70. Watch how the model answers in the first few rounds.

Round one, the model answers 73. Wrong. We record a score of −1.

Round two, the model answers 68. Still wrong. Another −1.

In some later round, the model answers 70. Correct. This time we record +1.

The rule is: add a point for a correct answer, subtract a point for a wrong one. The model repeatedly adjusts its parameters following these scores, pushing down the writings that were wrong and lifting the writings that were right, and gradually learns this kind of problem. That score is a `reward`, and this way of learning to do the right thing under the guidance of reward is `reinforcement learning`.

It is not the same as majority vote in Lecture 2. Majority vote is repeated sampling that amplifies ability the model already has; reinforcement learning changes the ability itself. Changing inference and changing training do not conflict, and they can be stacked: train the model stronger at train time, then amplify again at inference time with self-consistency voting.

By the end of this lecture we will have built a reinforcement-learning loop: the model generates reasoning and an answer, a program judges right or wrong and assigns a reward, and the reward is used to update parameters so accuracy rises round by round. We first work the ledger of each round by hand, then build the loop in code, and finally watch accuracy converge in an experiment.

## 1. Train-time vs test-time scaling

This section covers two points: how train-time and test-time scaling differ, and how much test-time scaling can amplify. We take the first point first, then use majority vote — which can be computed exactly — for the second.

Test-time scaling changes the sampling distribution. If the model's single-try accuracy on a problem is p, repeating the sample n times and taking the majority is sampling several times from the same distribution, and the result is still constrained by p. Train-time scaling changes the model parameters so that p itself rises. Every method in the first five lectures runs in the first dimension; this lecture enters the second.

The numerical behavior of majority vote can be computed exactly. Consider a concrete case: a model with single-try accuracy p answers independently n times, and majority vote counts as correct only when more than half of the tries are correct. The probability of "exactly k correct in n independent trials" is given by the binomial distribution; summing the probabilities from k = n/2 through n is the majority-vote accuracy. That sum is the tail probability of the binomial. We evaluate the formula next, and watch the boundary of test-time scaling as p takes different values.

In [ ]:
# Numerical experiment: majority vote can only amplify existing ability
import numpy as np
import matplotlib.pyplot as plt
from math import comb

get_ipython().run_line_magic('matplotlib', 'inline')

def majority_vote_accuracy(p, n):
    """Majority-vote accuracy of a model with single-try accuracy p over n independent samples."""
    total = 0.0
    for k in range(n // 2 + 1, n + 1):
        total += comb(n, k) * (p ** k) * ((1 - p) ** (n - k))
    return total

n = np.arange(1, 33)
for p in [0.3, 0.45, 0.55, 0.7]:
    ys = [majority_vote_accuracy(p, ni) for ni in n]
    plt.plot(n, ys, label=f"p={p:.2f}")

plt.axhline(0.5, color="gray", ls="--", lw=1)
plt.xlabel("number of samples n")
plt.ylabel("majority vote accuracy")
plt.legend()
plt.title("Test-time scaling amplifies existing ability only")
plt.show()

print("Key observation: when single-try accuracy p<=0.5, majority-vote accuracy does not exceed p,")
print("and can even fall as the sample count grows; only when p>0.5 does voting raise accuracy.")
print("Test-time scaling cannot create ability; raising p requires train-time scaling.")


## 2. Using successful reasoning to drive the next round of training

This section addresses how a model can learn to write reasoning when there is no ready-made "question → reasoning → answer" training data. The direct method is to hire people to annotate, but reasoning traces are hard to construct by hand: human annotation is expensive, and copying existing solutions does not cover new problem types. STaR (Self-Taught Reasoner) takes another route: let the model generate reasoning itself, and use "whether the answer is correct" as a filter.

Watch how the loop turns. The current model generates reasoning steps and an answer for each question; samples whose answers are correct go into dataset D_n. For samples that were generated wrong, the correct answer is placed in the prompt as a hint, and the model works backward to the reasoning — this step is called rationalization — producing another set D_rat. At training time the hint is not placed in the prompt, so the rationalized reasoning looks as if the model thought of it itself. The two sets are merged and used for fine-tuning, then the fine-tuned model generates, filters, and fine-tunes again. The loop continues, and the model gradually solves harder problems. The whole process needs only the questions and gold answers already in the dataset.

This practice of "training on one's own outputs" is called `bootstrapping`. That STaR works can be read as an approximation to the policy gradient. The intuition of the policy gradient is direct: the model generating a piece of reasoning is making one choice among many possibilities; if that choice brings a good result (a correct answer), raise the probability of making that kind of choice, and if it brings a bad result, lower it. In the math, think of the model as a two-stage generator: first write reasoning r, then give answer y, with overall probability p(y|x) = Σ_r p(r|x) p(y|x, r). Use the indicator 1(ŷ = y) as the reward — 1 for a correct answer, 0 for a wrong one — and the gradient of expected reward with respect to the parameters is exactly the form given by the log-derivative trick. Filtering out wrong rationales is equivalent to dropping the gradient of samples whose indicator is zero. Keep the intuition: under the signal "the answer is correct," the model keeps pushing up the probability of producing correct reasoning.

In a real implementation, each round fine-tunes again from the original pretrained model, to avoid overfitting to bootstrapped data. The toy demo below, to make the loop dynamics visible, uses a cumulative dataset to simulate ability growth. We build a fake model that can only do some addition: its accuracy is high on difficulty one (one-digit addition), not high on difficulty two (two-digit with carry), and zero on difficulty three (three-digit with carry). The STaR loop is meant to show how the dataset expands round by round, how rationalization supplies samples that failed generation but can be worked backward, and how hard problems unlock gradually.

In [ ]:
# STaR toy arithmetic task and rule-based generator
import numpy as np

np.random.seed(42)

def difficulty(a, b):
    """Difficulty of addition problem (a, b): three bins by the number of digits of the larger addend."""
    if max(a, b) < 10:
        return 1
    if max(a, b) < 100:
        return 2
    return 3

class ToyArithModel:
    """A small model that can do only some addition. skill records accuracy in each difficulty bin."""

    def __init__(self, skill):
        self.skill = dict(skill)

    def generate(self, x):
        """Generate (rationale, answer, whether correct) for problem x=(a, b)."""
        a, b = x
        d = difficulty(a, b)
        y = a + b
        if np.random.rand() < self.skill[d]:
            return f"Add {a} and {b} place by place, carrying from ones to tens.", y, True
        ans = y + np.random.choice([-3, -2, -1, 1, 2, 3])   # a wrong answer is never equal to the correct one
        return f"Add {a} and {b} place by place, carrying from ones to tens.", ans, False

    def rationalize(self, x, hint):
        """Using the correct answer as a hint, work backward to a rationale and give an answer."""
        a, b = x
        d = difficulty(a, b)
        rat_skill = {1: 0.97, 2: 0.85, 3: 0.55}
        y = a + b
        ans = y if np.random.rand() < rat_skill[d] else (y + np.random.choice([-2, -1, 1, 2]))
        rationale = f"Add {a}+{b} place by place and handle the carry."
        return rationale, ans

model = ToyArithModel({1: 0.95, 2: 0.30, 3: 0.0})
print("Initial skill (accuracy by difficulty):", model.skill)
print("Sample generation:", model.generate((23, 47)))


In [ ]:
# STaR outer loop: generate → filter → rationalize → merge
def star_round(questions, model):
    """One round of STaR generation and filtering.

    Return (D_n, D_rat): D_n is samples that were generated correctly, D_rat is samples
    that were generated wrong but rationalized correctly with a hint.
    Format of both is (x, rationale, y).
    """
    D_n, D_rat = [], []
    for x in questions:
        rationale, ans, ok = model.generate(x)
        y = x[0] + x[1]
        if ok and ans == y:
            D_n.append((x, rationale, y))
        else:
            rationale_rat, ans_rat = model.rationalize(x, y)
            if ans_rat == y:
                D_rat.append((x, rationale_rat, y))
    return D_n, D_rat

def simulate_finetune(model, train_data):
    """Simulate fine-tuning: each correct rationale seen raises that difficulty bin's accuracy by 0.01, capped at 0.95."""
    seen = {}
    for (a, b), _, _ in train_data:
        d = difficulty(a, b)
        seen[d] = seen.get(d, 0) + 1
    for d, cnt in seen.items():
        model.skill[d] = min(0.95, model.skill[d] + cnt * 0.01)
    return model


This subsection tracks how the dataset expands round by round inside one STaR loop. We compute the counts at each step by hand, so that when the code runs later, each printed number has a known source.

The code above built the fake model and the loop functions. We first work one round by hand and watch the dataset expand. Initial accuracies are {1: 0.95, 2: 0.30, 3: 0.0}, with 20 problems per bin.

Step one, generate and filter. The model generates one rationale and answer per problem; correct answers go into D_n:

- Difficulty one (one-digit): 20 problems × 0.95 ≈ 19 into D_n
- Difficulty two (two-digit with carry): 20 × 0.30 = 6 into D_n
- Difficulty three (three-digit with carry): 20 × 0.00 = 0 into D_n

Step two, rationalization. Samples that failed generation are worked backward with the correct answer as a hint; rationalization accuracies are {1: 0.97, 2: 0.85, 3: 0.55}:

- Difficulty one: about 1 failure, 1 × 0.97 ≈ 1 into D_rat
- Difficulty two: 14 failures, 14 × 0.85 ≈ 12 into D_rat
- Difficulty three: all 20 failed, 20 × 0.55 ≈ 11 into D_rat

Step three, merge and fine-tune. Train on D_n and D_rat together; each correct rationale raises that difficulty bin's accuracy by 0.01, capped at 0.95:

- Difficulty one: about 19 + 1 = 20 samples, 0.95 plus 0.20 still caps at 0.95
- Difficulty two: about 6 + 12 = 18 samples, 0.30 + 0.18 = 0.48
- Difficulty three: about 0 + 11 = 11 samples, 0.00 + 0.11 = 0.11

At the start of round two, difficulty-three accuracy has moved from 0 to 0.11, and the model begins to generate a few correct three-digit solutions on its own. That is the meaning of bootstrapping: the model trains on its own outputs, unlocking a difficulty makes the model stronger, and a stronger model solves more problems, so the loop continues.

By contrast, if rationalization is turned off, the number of difficulty-three samples entering the training set in round one is 0, accuracy is still 0 at the start of round two, and difficulty three never enters the training set. Eval accuracy stops at (0.95 + 0.95 + 0)/3 ≈ 0.63; with rationalization on, all three bins can rise to 0.95, and eval accuracy is about 0.95. That is the contrast the next code cell is meant to show.

A caveat: the numbers above are expectations. In the actual code every step is a random sample, so a single round will fluctuate around these values, but the direction of the loop is determined.

In [ ]:
# Run several STaR rounds, comparing rationalization on vs off
import matplotlib.pyplot as plt

def make_questions(rng, n1, n2, n3):
    """Generate n1 one-digit, n2 two-digit, and n3 three-digit addition problems."""
    qs = []
    for _ in range(n1):
        qs.append((rng.randint(2, 9), rng.randint(2, 9)))
    for _ in range(n2):
        qs.append((rng.randint(15, 49), rng.randint(15, 49)))
    for _ in range(n3):
        qs.append((rng.randint(150, 499), rng.randint(150, 499)))
    return qs

questions = make_questions(np.random.RandomState(7), 20, 20, 20)
eval_questions = make_questions(np.random.RandomState(11), 10, 10, 10)

def run_star(rounds, use_rationalization):
    """Run the STaR outer loop; return per-round (train-set size, eval accuracy)."""
    np.random.seed(0)
    model = ToyArithModel({1: 0.95, 2: 0.30, 3: 0.0})
    train_data = []
    sizes, accs = [], []
    for _ in range(rounds):
        D_n, D_rat = star_round(questions, model)
        train_data.extend(D_n)
        if use_rationalization:
            train_data.extend(D_rat)
        simulate_finetune(model, train_data)
        acc = np.mean([model.skill[difficulty(a, b)] for a, b in eval_questions])
        sizes.append(len(train_data))
        accs.append(round(acc, 4))
    return sizes, accs

rounds = 9
sizes_rat, accs_rat = run_star(rounds, use_rationalization=True)
sizes_no, accs_no = run_star(rounds, use_rationalization=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(range(1, rounds + 1), sizes_rat, marker="o", label="with rationalization")
axes[0].plot(range(1, rounds + 1), sizes_no, marker="s", label="without")
axes[0].set_xlabel("round")
axes[0].set_ylabel("train data size")
axes[0].set_title("Dataset growth")
axes[0].legend()
axes[1].plot(range(1, rounds + 1), accs_rat, marker="o", label="with rationalization")
axes[1].plot(range(1, rounds + 1), accs_no, marker="s", label="without")
axes[1].set_xlabel("round")
axes[1].set_ylabel("eval accuracy")
axes[1].set_title("Eval accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()

print("Train-set size each round with rationalization on:", sizes_rat)
print("Train-set size each round with rationalization off:", sizes_no)
print("Eval accuracy each round with rationalization on:", accs_rat)
print("Eval accuracy each round with rationalization off:", accs_no)
print("Key observation: without rationalization, not a single three-digit sample is generated correctly,")
print("that difficulty never enters the training set, and accuracy stops at 0.63;")
print("rationalization works backward from the correct answer, adds failed samples to the training set,")
print("hard problems unlock gradually, and accuracy rises to 0.95.")


In [ ]:
# Generate a rationale with an LLM (live mode uses the API; a live API demo uses a deterministic branch)
import sys, os, re
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm

client = get_llm()
print("Client is a live API demo:", False)

def generate_rationale(client, a, b):
    """Ask the LLM for one (rationale, answer); parse the last number in the reply as the answer."""
    prompt = f"Compute {a} plus {b}. Write the reasoning first, then give the answer."
    reply = client.chat([{"role": "user", "content": prompt}])
    nums = re.findall(r"-?\d+", reply)
    ans = int(nums[-1]) if nums else None
    return reply, ans

for a, b in [(23, 47), (7, 5)]:
    reply, ans = generate_rationale(client, a, b)
    verdict = "correct" if ans == a + b else "incorrect"
    print(f"Reply for problem {a}+{b}: {reply}")
    print(f"  parsed answer {ans}, true answer {a+b}, verdict {verdict}")


## 3. Training reasoning ability with within-group comparison

This section upgrades the reward from the two values "right or wrong" to a continuous score, and makes training cheaper in GPU memory. STaR in the previous section drove bootstrapping with a binary signal, and that signal is coarse: it can only tell right from wrong, and cannot express intermediate cases such as "wrong, but the direction is close." DeepSeekMath switches to a continuous reward model and updates the policy with PPO, a mature algorithm in reinforcement learning. PPO has two costs: the value network is the same size as the policy, so memory doubles; the reward is given only at the last token, so per-token values are hard to train accurately. GRPO is designed to remove those two costs.

Look first at the PPO objective, then unpack what each symbol is saying.

$$J_{PPO}(\theta) = \mathbb{E}_{q,o}\big[ \tfrac{1}{|o|} \sum_t \min( r_t(\theta) A_t,\, \text{clip}(r_t(\theta),\, 1-\varepsilon,\, 1+\varepsilon) A_t ) \big]$$

$r_t(\theta)$ is the importance ratio; it compares the probability that the new and old policies assign to the token at position t: new policy $\pi_\theta$ divided by old policy $\pi_{\theta_\text{old}}$. $A_t$ is advantage, meaning "how much better this position is than the average expectation," computed by GAE from a learned value function $V_\psi$. min plus clip limits the size of a single update, to keep one change from being too large. A KL penalty is added into the per-step reward, $r_t = r_\varphi(q, o_{\le t}) - \beta \log(\pi_\theta / \pi_\text{ref})$, which keeps the new policy from drifting too far from the reference policy.

GRPO's key move is to drop the value network and use within-group statistics instead. For each question, sample G outputs as a group; advantage is determined entirely by relative reward inside the group:

$$\tilde{r}_i = \frac{r_i - \text{mean}(r)}{\text{std}(r)}$$

Giving a reward that looks only at the final answer, not at intermediate steps, is called outcome supervision; outcome supervision assigns this normalized reward to every token in the output. The KL penalty is moved out of the reward and into the objective, becoming $-\beta\, D_\text{KL}(\pi_\theta \| \pi_\text{ref})$, with an unbiased estimator that is guaranteed non-negative. The two sets of formulas have the same structure; the differences are only the source of advantage and the placement of KL. Component comparison:

| Component | PPO | GRPO |
|:---|:---|:---|
| Value network | required, same size as the policy | not required |
| Source of advantage | GAE + value function | within-group statistics (r−mean)/std |
| Placement of KL penalty | inside the per-step reward | inside the objective |
| Samples per question | 1 output | G outputs as a group |
| Extra GPU memory | policy + value, two models | policy only |

This subsection explains what the value network in PPO does, and why GRPO can drop it entirely.

PPO uses advantage A_t to tell the model "how much better this position is than expected." It uses a value network V_psi of the same size as the policy to predict, token by token, "how much reward can be obtained from now until the end," then GAE combines those predictions into A_t. That design has two costs: the value network and the policy each occupy a copy of GPU memory; the reward is given only at the end, so the per-token value target is sparse, and the value network is hard to train accurately.

GRPO takes another route: instead of predicting a value for each token, it samples G complete outputs for the same question and uses the mean reward of that group as a baseline. The advantage of the i-th output in the group is determined only by its relative position inside the group:

$$\tilde{r}_i = \frac{r_i - \text{mean}(r)}{\text{std}(r)}$$

The design can be read as follows. For a fixed question q, the group mean mean(r) is an estimate of "how much the current model can obtain on this problem on average," which is this problem's own baseline. If a single output is above the group mean, it is better than the model's average, advantage is positive, and the model should learn that way of doing it; if it is below the group mean, advantage is negative, and the model should move away from that way of doing it. The learning signal changes from "absolute score" to "relative rank."

Side by side, GRPO replaces the value network's prediction with statistics of a group of real samples: there is no longer a need for a per-token long-horizon return estimate, and no longer a need for a second copy of model weights. The cost is that each question must sample G outputs, moving compute from "train a value network" to "sample a few more times." For math problems whose answers can be verified, DeepSeekMath uses G=64; the toy code in this section uses G=32.

This subsection walks the advantage formula one step at a time. We work the formula by hand, compute each number, and see where the signs come from, so that the code later is readable with those numbers in mind.

Walk the formula on a group of 4 outputs. Let the rewards of G=4 outputs on one problem be

$$r = [1, -1, 0, 1]$$

that is, two outputs correct (+1), one incorrect (−1), and one partly correct (0). Advantage is computed in three steps.

Step one, the group mean. Add the 4 rewards and divide by 4:

$$\text{mean}(r) = \frac{1 + (-1) + 0 + 1}{4} = \frac{1}{4} = 0.25$$

Step two, the group standard deviation. Take the difference of each reward from the mean, square, sum, divide by 4, and take the square root. The standard deviation here is the population standard deviation (ddof=0), dividing by G rather than G−1, matching the paper:

| Sample i | r_i | r_i − mean | (r_i − mean)² |
|:---|:---|:---|:---|
| 1 | 1 | 0.75 | 0.5625 |
| 2 | −1 | −1.25 | 1.5625 |
| 3 | 0 | −0.25 | 0.0625 |
| 4 | 1 | 0.75 | 0.5625 |

Sum of squares = 0.5625 + 1.5625 + 0.0625 + 0.5625 = 2.75, variance = 2.75 / 4 = 0.6875, standard deviation = √0.6875 ≈ 0.8292.

Step three, subtract the mean from each reward and divide by the standard deviation:

$$\tilde{r}_i = \frac{r_i - 0.25}{0.8292} \Rightarrow \tilde{r} = [0.9045, -1.5076, -0.3015, 0.9045]$$

The two correct outputs have the same positive advantage, the incorrect output is negative, and the partly correct output also receives a negative value — it is below the group mean of 0.25. The same advantage is assigned to every token in that output; that is outcome supervision.

Why subtract the mean. A positive mean means this problem is not hard for the current model. Without subtracting the mean, the incorrect output's reward −1 would still raise its probability, even though it is already below the group average; after subtracting the mean, advantage is meaningful only relative to the group average, so correct answers are pushed up and incorrect answers are pushed down. From the policy-gradient view, subtracting a constant that does not depend on the action does not change the expected gradient, but it lowers the variance of the gradient estimate — the classic baseline trick in REINFORCE.

Why divide by the standard deviation. The numeric range of rewards varies by problem: on some problems rewards cluster near 0, on others they spread out. Dividing by the standard deviation turns advantage into "how many group standard deviations from the group mean," so update sizes are comparable across problems and rounds, and a single learning rate can be used. The standard deviation roughly estimates "how hard it is on average to score on this problem"; dividing by it normalizes for difficulty.

One boundary that must be noted: if every reward in the group is the same (all correct or all incorrect), the standard deviation is 0 and the formula becomes 0/0. The next code cell fills that case with 0, which is also the problem Dynamic Sampling in DAPO is meant to handle.

In [ ]:
# Hand calculation of GRPO within-group advantage
import numpy as np

r = np.array([1.0, -1.0, 0.0, 1.0])   # G=4 outputs: two correct (+1), one wrong (-1), one partial (0)
mean_r = r.mean()
std_r = r.std(ddof=0)                 # population std, matching the paper
adv = (r - mean_r) / std_r

print("within-group reward r        :", r)
print("within-group mean            :", round(mean_r, 4))
print("within-group std             :", round(std_r, 4))
print("normalized advantage         :", np.round(adv, 4))
print("Key observation: correct samples have positive advantage, incorrect samples negative,")
print("and a partly correct sample also receives a negative value, because it is below the group mean.")


This subsection explains what lets GRPO update the policy boldly without taking too large a step. The answer is a constraint term, the KL penalty. We first say what it is, then work a few values by hand, and finally see why it is always at least 0.

The GRPO objective has two parts: raise the reward, and a constraint that keeps the policy from chasing reward too fast. The constraint is the KL distance between a reference policy π_ref (usually the pretrained model, or the policy used at sampling time) and the current policy π_θ, multiplied by a coefficient β. In one sentence: change is allowed, but not too far from the starting place.

The KL term sits in the objective and acts as a fine. If the new policy pushes a token's probability away from the reference, a penalty is applied; the farther the push, the larger the penalty. During training the model balances "push probability toward high-reward tokens" against "do not drift too far from the reference"; larger β is more conservative.

GRPO uses a first-order unbiased estimator (Schulman 2020). Write log_ratio = log(π_θ / π_ref); when outputs are sampled from π_θ, its expectation is an unbiased estimate of KL:

$$\hat{D}_\text{KL} = \exp(-\text{log\_ratio}) + \text{log\_ratio} - 1$$

Work a few values token by token to see the behavior:

| log_ratio | Meaning | Estimate |
|:---|:---|:---|
| 0 | π_θ identical to π_ref | exp(0) + 0 − 1 = 0 |
| +0.1 | that token's probability rose about 10% | 0.9048 + 0.1 − 1 = 0.0048 |
| −0.1 | that token's probability fell about 10% | 1.1052 − 0.1 − 1 = 0.0052 |
| +0.693 | that token's probability doubled | 0.5 + 0.693 − 1 = 0.193 |

When policy and reference are identical the estimate is 0; the farther they drift, the larger the estimate, so it is measuring "how far apart the two distributions are." At β = 0.04, a token whose probability doubled contributes about 0.04 × 0.193 = 0.0077 of penalty; a 10% move contributes only about 0.0002.

Why this expression is always at least 0. The graph of exp(t) lies everywhere above the line 1 + t, that is exp(t) ≥ 1 + t for every real t. Set t = −log_ratio, then exp(−log_ratio) ≥ 1 − log_ratio, and rearranging gives exp(−log_ratio) + log_ratio − 1 ≥ 0. Every token's contribution is non-negative; the penalty appears only when the policy drifts, and does not tax an update that was already correct.

Compared with PPO, PPO places the penalty inside the per-step reward (r_t = r_t − β log(π_θ/π_ref)); GRPO places it in the outer objective. The placement differs, the role is the same: limit the size of a single update, and keep the policy balanced between reward and stability.

In [ ]:
# GRPO objective from scratch (including the unbiased KL estimator)
import numpy as np

def unbiased_kl(log_ratio):
    """Unbiased KL estimate of π_θ relative to π_ref: exp(-log_ratio) + log_ratio - 1.

    log_ratio = log(π_θ / π_ref). When π_θ = π_ref, log_ratio = 0 and the estimate is 0;
    the estimate is guaranteed non-negative (Schulman 2020).
    """
    return np.exp(-log_ratio) + log_ratio - 1.0

def grpo_loss(ratios, adv_per_output, lengths, eps=0.2, beta=0.04, log_ratios=None):
    """Return the GRPO loss of a group of outputs (negative objective + KL penalty).

    ratios: [G] one mean importance ratio per output; adv_per_output: [G] within-group advantage;
    lengths: [G] token count of each output; log_ratios: log-ratios of the same shape as ratios, used for KL.
    """
    G = len(ratios)
    clipped = np.minimum(
        ratios * adv_per_output,
        np.clip(ratios, 1 - eps, 1 + eps) * adv_per_output,
    )
    mean_loss = (clipped / lengths).sum() / G
    kl_penalty = beta * np.mean(unbiased_kl(log_ratios)) if log_ratios is not None else 0.0
    return -mean_loss + kl_penalty

print("Unbiased KL estimate: at log_ratio=0 =", unbiased_kl(0.0),
      ", at +0.1 =", round(unbiased_kl(0.1), 4),
      ", at -0.1 =", round(unbiased_kl(-0.1), 4))

ratios = np.array([1.1, 0.9])        # policy slightly off the distribution used at sampling
adv = np.array([1.0, -1.0])          # within-group advantage
lengths = np.array([5, 5])
log_ratios = np.array([0.1, -0.1])
loss = grpo_loss(ratios, adv, lengths, beta=0.04, log_ratios=log_ratios)
print("GRPO loss (with KL penalty):", round(loss, 4))
print("Key observation: the KL penalty constrains updates that drift from the reference model,")
print("so the policy does not move too far from the reference in one step.")


In [ ]:
# All-correct group: advantage=0, gradient vanishes
import numpy as np

def group_advantage(r):
    """Within-group normalized advantage (r-mean)/std; if std=0 set 0 to avoid 0/0."""
    mean = r.mean()
    std = r.std(ddof=0)
    with np.errstate(divide="ignore", invalid="ignore"):
        adv = (r - mean) / std
    return np.nan_to_num(adv, nan=0.0, posinf=0.0, neginf=0.0)

all_correct = np.array([1.0, 1.0, 1.0, 1.0])   # all-correct group
all_wrong   = np.array([-1.0, -1.0, -1.0, -1.0])
mixed       = np.array([1.0, 1.0, -1.0, -1.0])

for name, r in [("all-correct", all_correct), ("all-wrong", all_wrong), ("half-and-half", mixed)]:
    adv = group_advantage(r)
    grad_scale = np.abs(adv).sum()   # proportional to gradient norm
    print(f"{name} group: mean={r.mean():.2f} std={r.std(ddof=0):.2f} "
          f"advantage={np.round(adv, 3)} gradient scale={grad_scale:.2f}")

print("Key observation: in an all-correct or all-wrong group, every advantage is 0, so the contributed gradient is 0.")
print("As the model grows stronger, all-correct groups become more common and the share of effective samples falls; that is the sampling problem DAPO addresses.")


In [ ]:
# PPO/GRPO clipped objective: the shape of the trust region
import numpy as np
import matplotlib.pyplot as plt

def clipped_surrogate(ratio, advantage, eps=0.2):
    """Clipped objective min(ratio*A, clip(ratio, 1-eps, 1+eps)*A)."""
    return np.minimum(ratio * advantage,
                      np.clip(ratio, 1 - eps, 1 + eps) * advantage)

ratio = np.linspace(0.0, 2.0, 201)
eps = 0.2
for A in [1.0, -1.0]:
    plt.plot(ratio, clipped_surrogate(ratio, A, eps), label=f"A={A:+g}")

plt.axvline(1 - eps, color="gray", ls="--", lw=1)
plt.axvline(1 + eps, color="gray", ls="--", lw=1)
plt.xlabel("importance ratio pi_theta / pi_old")
plt.ylabel("clipped surrogate objective")
plt.legend()
plt.title("Clipped objective forms a trust region")
plt.show()

print("Key observation: when A>0 the objective caps after the ratio exceeds 1+eps, preventing a single oversized step;")
print("when A<0 the objective caps after the ratio falls below 1-eps, preventing probability from being driven to 0.")


In [ ]:
# Advantage propagation: outcome supervision vs process supervision
import numpy as np

step_rewards = np.array([[1.0, 1.0], [-1.0, -1.0]])   # [output, step]
outcome_rewards = np.array([1.0, -1.0])                # overall reward of each output
T = step_rewards.shape[1]

# Outcome supervision: one advantage for the whole output
mean_o = outcome_rewards.mean()
std_o = outcome_rewards.std(ddof=0)
adv_outcome = (outcome_rewards - mean_o) / std_o

# Process supervision: normalize each step within the group, then accumulate from that step onward by token position
normalized = (step_rewards - step_rewards.mean(axis=0)) / step_rewards.std(axis=0, ddof=0)
adv_process = np.zeros_like(normalized)
for t in range(T):
    adv_process[:, t] = normalized[:, t:].sum(axis=1)

print("Step rewards (2 outputs × 2 steps):")
print(step_rewards)
print("Outcome-supervision per-token advantage:", np.round(adv_outcome, 3))
print("Process-supervision per-token advantage:", np.round(adv_process, 3))
print("Key observation: outcome supervision gives the whole output the same advantage;")
print("process supervision splits the reward by step, so earlier tokens accumulate reward from more steps.")


In [ ]:
# Unified paradigm: data source, reward, gradient coefficient
import numpy as np

def gradient_coefficient(method, reward, mean_reward=None, std_reward=None):
    """Gradient coefficient GC(q,o,t,π_ref) in the unified paradigm. SFT/RFT coefficients are fixed; GRPO varies with reward magnitude."""
    if method in ("SFT",):
        return 1.0
    if method in ("RFT", "Online RFT"):
        return float(reward > 0)   # keep only the correct ones; wrong samples get coefficient 0
    if method == "GRPO":
        return (reward - mean_reward) / std_reward
    raise ValueError(method)

rewards = np.array([1.0, 1.0, -1.0, 1.0])   # rewards of a group of 4 outputs: three correct, one wrong
mean_r = rewards.mean()
std_r = rewards.std(ddof=0)

methods = ["SFT", "RFT", "Online RFT", "GRPO"]
print(f"{'method':<12}" + "".join(f"{'out '+str(i):>8}" for i in range(4)))
for m in methods:
    coefs = [gradient_coefficient(m, r, mean_r, std_r) for r in rewards]
    print(f"{m:<12}" + "".join(f"{c:>9.3f}" for c in coefs))

print("Key observation: RFT sets wrong samples to 0 (neither learn nor penalize);")
print("GRPO penalizes wrong samples by magnitude (negative coefficient) and weights positive samples by within-group relative position.")
print("The unified paradigm puts SFT/RFT/Online RFT/GRPO in one framework;")
print("the differences are only the data source (offline/online) and the gradient coefficient.")


## 4. Stability problems in large-scale reinforcement learning

This section covers which stability problems appear when the formulas of the previous section are scaled to training on thousands of GPUs, and how to fix them. With naive GRPO on Qwen2.5-32B the paper reached only 30 points on AIME 2024, while the matching DeepSeek-R1-Zero setup is 47 points. The gap is not in the formula, but in the engineering details missing between the paper formula and stable large-scale training. DAPO (Decoupled Clip and Dynamic sAmpling Policy Optimization) supplies those details, four tricks in all.

Each trick matches a failure mode. The first failure is entropy collapse: the model becomes more deterministic as training proceeds, low-probability tokens are held down by the clip upper bound, and the exploration space shrinks. The matching trick is Clip-Higher. The second failure is zero gradient on all-correct groups: as the model grows stronger, a group of outputs is often all correct, and that group has no gradient. The matching trick is Dynamic Sampling. The third failure is diluted weight on long samples: sample-level loss lowers the weight of a single token inside a long sample. The matching trick is Token-Level Policy Gradient Loss. The fourth failure is wrongly penalizing truncated samples: an output that exceeds the length cap is truncated, and even correct reasoning is penalized. The matching trick is Overlong Reward Shaping. Each is demonstrated numerically below.

Separately, DAPO drops the KL penalty entirely: the distribution of a long-CoT model already differs a lot from the reference, so a KL constraint is unnecessary. Advantage is still the within-group statistic Â_{i,t} = (R_i − mean({R_i})) / std({R_i}), and the reward is a rule reward: +1 if the answer is equivalent, otherwise −1; no learned reward model is needed. The differences from GRPO concentrate on the normalization factor (sample-level changed to token-level) and the decoupling of the upper and lower clip bounds.

This subsection starts with the first failure mode: entropy collapse. Understanding it requires a quantity that measures "how certain the model is"; that quantity is entropy. We first compute entropy on a small K=4 example, then look at its relation to Clip-Higher.

Entropy measures how spread out a distribution is. For the distribution at one token position, entropy is defined as

$$H = -\sum_i p_i \log p_i$$

When probability concentrates on one token, entropy is 0; a uniform distribution has maximum entropy. Compute a K=4 candidate-token example. The uniform (0.25, 0.25, 0.25, 0.25) has entropy log 4 ≈ 1.386; concentrated to (0.97, 0.01, 0.01, 0.01), entropy is about −0.97 log 0.97 − 3 × 0.01 log 0.01 ≈ 0.168. Early in training the distribution is still spread out; through mid-to-late training entropy falls steadily, meaning the model is becoming more certain.

Falling entropy in RL training is itself normal — the correct token should be selected. The problem is falling too fast. Entropy of zero means the model recognizes only one token at every position and no longer tries other possibilities; for a reasoning model, that is giving up the ability to explore new solutions, and performance stops at the currently best behavior, unable to keep rising. That is what the paper calls entropy collapse.

DAPO's first move, Clip-Higher, targets exactly this mechanism. The clipped objective caps how far a single step can raise a probability, at a factor of 1 + ε_high. The key is whom that upper bound applies to: a token whose probability is already near 1 cannot exceed 1, so the bound does not bind; low-probability tokens are the ones it holds down. A token with p = 0.01 can rise in one step at most to 0.01 × 1.2 = 0.012 (when ε = 0.2) or 0.01 × 1.28 = 0.0128 (when ε_high = 0.28). Those low-probability tokens are exactly the carriers of exploration — they may correspond to longer, more different reasoning paths. The default 0.2 seals their room to rise too tightly, and the distribution can only concentrate more and more on the current winner.

Raising the upper bound from 0.2 to 0.28 raises the allowed single-step growth from 20% to 28%, a factor of 1.4, which opens relatively more room for low-probability tokens to grow, while high-probability tokens are unaffected. The absolute extra rise in one step is still small, but accumulated over time the model keeps room to try new tokens, and entropy does not hit zero too early. The next code cell plots the effect of these two upper bounds at different probabilities.

In [ ]:
# Clip-Higher: room for low-probability tokens to rise
import numpy as np
import matplotlib.pyplot as plt

def max_reachable(p, eps_high):
    """Hard upper bound on probability after one update for a token currently at p: p*(1+eps_high) (capped at 1)."""
    return np.minimum(1.0, p * (1 + eps_high))

p = np.linspace(0.001, 0.9, 200)
for eps_high in [0.20, 0.28]:
    plt.plot(p, max_reachable(p, eps_high), label=f"eps_high={eps_high}")

plt.plot(p, p, "k--", lw=1, label="no update")
plt.xlabel("current token prob p")
plt.ylabel("max prob after one update")
plt.legend()
plt.title("Clip-Higher leaves room for low-prob tokens")
plt.show()

for pi in [0.01, 0.1, 0.9]:
    print(f"p={pi:.2f}: ε=0.20 upper bound {max_reachable(pi, 0.20):.4f}, "
          f"ε_high=0.28 upper bound {max_reachable(pi, 0.28):.4f}")

print("Key observation: an exploration token at p=0.01 barely moves in one step, and the distribution tends toward a single peak (entropy collapse);")
print("a token at p=0.9 is not limited by the upper bound (it already caps at 1).")
print("Raising ε_high to 0.28 leaves relatively more room for low-probability tokens to rise, preserving exploration.")


This subsection covers the second failure mode: an all-correct group has no gradient, and how DAPO filters it out with Dynamic Sampling. We first compute how common an "all-correct group" is, then how much compute filtering saves.

Within-group advantage has a hidden defect: when every output in a group is the same (all correct or all incorrect), the standard deviation is 0, every advantage is 0, and the group contributes nothing to the gradient. The longer training runs and the stronger the model, the more all-correct groups there are, and the more compute is wasted.

Quantify the waste. Let the model's single-try accuracy be p and sample G outputs per group. The probability that a group is mixed (at least one correct and at least one incorrect, so advantage has both signs) is

$$\Pr(\text{mixed}) = 1 - p^G - (1-p)^G$$

Substitute G=8 at a few values of p:

| Single-try accuracy p | All-correct share p^G | Mixed-group share |
|:---|:---|:---|
| 0.2 | 0.2⁸ ≈ 0.000 | 1 − 0.000 − 0.168 ≈ 0.832 |
| 0.5 | 0.5⁸ ≈ 0.004 | 1 − 0.004 − 0.004 ≈ 0.992 |
| 0.8 | 0.8⁸ ≈ 0.168 | 1 − 0.168 − 0.000 ≈ 0.832 |
| 0.98 | 0.98⁸ ≈ 0.851 | 1 − 0.851 − 0.000 ≈ 0.149 |

At p=0.5 almost every group is mixed, so the gradient signal is plentiful; after the model strengthens to p=0.98, 85% of groups are all correct, and only about 15% of groups still produce a useful gradient. For a model that is already strong but still needs polishing, that means most samples in a training round are spinning in place.

DAPO's Dynamic Sampling is direct: sample extra outputs per question, keep only mixed groups (groups that contain both correct and incorrect) before the update, drop all-correct and all-wrong groups, then recompute advantage on the kept groups with a fresh normalization. Every sample in the batch then has a non-zero gradient, and late training no longer spends time on problems the model has already mastered.

In [ ]:
# Dynamic Sampling: all-correct / all-wrong groups have no gradient
import numpy as np
import matplotlib.pyplot as plt

def effective_fraction(p, G):
    """Share of mixed groups (0<correct<G) when single-try accuracy is p and G samples are drawn per question."""
    return 1.0 - p ** G - (1 - p) ** G

G = 8
rounds = np.arange(0, 12)
p_curve = 0.2 + 0.78 * rounds / rounds[-1]   # simulate the model strengthening with training, p from 0.2 to 0.98
eff = [effective_fraction(pi, G) for pi in p_curve]

plt.plot(rounds, eff, marker="o")
plt.axhline(0.5, color="gray", ls="--", lw=1)
plt.xlabel("RL training round")
plt.ylabel("fraction of effective (mixed) groups")
plt.title("Effective groups shrink as the model improves")
plt.show()

for rnd, pi, e in zip(rounds, p_curve, eff):
    print(f"round {rnd:2d}: p={pi:.2f}, mixed-group share={e:.3f}")

print("Key observation: late in training p approaches 1, most groups have all G outputs correct,")
print("advantage is 0 and the gradient is 0. Dynamic Sampling filters out all-correct / all-wrong prompts,")
print("keeps only mixed groups, and guarantees that every batch has a useful gradient.")


In [ ]:
# Sample-level vs token-level loss: a weight comparison
import numpy as np

lengths = np.array([10, 100])   # a short sample and a long sample
G = len(lengths)

sample_level = 1.0 / G / lengths          # sample-level: samples share 1/G, then that share is split by sample length
token_level = 1.0 / lengths.sum()         # token-level: split equally across all tokens of all samples

print("Sample-level: per-token weight of the short sample", round(sample_level[0], 4),
      ", per-token weight of the long sample", round(sample_level[1], 4))
print("Token-level: per-token weight of short/long samples", round(token_level, 4))
print(f"Under sample-level loss the long sample's single token is diluted {lengths.max() / lengths.min():.0f}×")
print("Key observation: sample-level loss shrinks the per-token weight inside long samples,")
print("so rambling patterns in long outputs do not receive the penalty they should; token-level normalization treats short and long samples alike.")


In [ ]:
# DAPO overlong reward shaping
import numpy as np

L_max, L_cache = 16384, 4096
thresh = L_max - L_cache   # 12288; penalty starts only beyond this length

def reward_length(length):
    """Piecewise function for soft overlong punishment."""
    if length <= thresh:
        return 0.0
    if length <= L_max:
        return (thresh - length) / L_cache
    return -1.0

for length in [5000, 12288, 14000, 16384, 20000]:
    print(f"len={length:5d} → R_length = {reward_length(length):+.4f}")

print("Key observation: ordinary length is not penalized; near the cap a linear soft penalty (easing from 0 down to -1),")
print("and past the cap a flat -1. Filter overlong samples first, then apply a smooth penalty,")
print("to avoid wrongly penalizing outputs whose reasoning is correct but too long.")


In [ ]:
# AIME stacked curve: how many points each trick is worth
import matplotlib.pyplot as plt

steps = ["Naive\nGRPO", "+Overlong\nFiltering", "+Clip-\nHigher",
         "+Soft Overlong\nPunishment", "+Token-\nlevel", "+Dynamic\nSampling"]
scores = [30, 36, 38, 41, 42, 50]

plt.bar(range(len(scores)), scores, color="steelblue")
plt.xticks(range(len(scores)), steps)
plt.ylabel("AIME 2024 score (avg@32)")
plt.title("DAPO: each trick adds a few points")
plt.show()

for s, sc in zip(["Naive GRPO", "+Overlong Filtering", "+Clip-Higher",
                  "+Soft Overlong", "+Token-level", "+Dynamic Sampling (DAPO)"], scores):
    print(f"{s:<28} {sc} points")


## 5. From reinforcement to Agent training

This section gathers the three papers and covers two points: what the complete recipe for train-time scaling is, and how it relates to later lectures and to the Agent framework as a whole.

Taken together, the three papers are the complete recipe for train-time scaling. STaR supplies the idea of bootstrapping: the model trains on its own correct reasoning, with no one writing the reasoning. GRPO turns bootstrapping into a scalable training algorithm: drop the value network, compute advantage from within-group relative reward, and use an unbiased estimator for the KL penalty. DAPO solves stable training after the formula is scaled up: clip bounds, sampling efficiency, token weights, and truncation rewards, four engineering details. Idea, algorithm, and engineering are three layers, matching a lecture's full path from motivation to system.

Back to the question of section 1, the two dimensions can be told apart in one sentence: test-time scaling changes the sampling distribution, train-time scaling changes the model parameters. Repeated sampling, voting, reranking, and search in the first five lectures all run under the same weights; the methods in this lecture change the weights themselves. The two routes can be combined: a model trained by DeepSeekMath, then voted with self-consistency at inference time, raises MATH accuracy from 51.7% to 60.9%.

For the Agent course, train-time scaling is the foundation of the lectures that follow. Lecture 7 lets an Agent design an Agent, Lecture 8 uses search to strengthen reasoning, and Lecture 9's post-training evolution all reuse this lecture's recipe: a verifiable reward, a group of samples, one policy update. The only difference is that the reward signal changes from "whether the answer is correct" to "whether the task is done," "whether the code runs," "whether the tool call succeeded." The model is no longer only learning to predict the next token; under the guidance of environment feedback it learns to complete tasks.

In [ ]:
# Train a tiny policy with torch: the GRPO update flow
import numpy as np
import torch
import torch.nn.functional as F

np.random.seed(0)
torch.manual_seed(0)

class ToyPolicy(torch.nn.Module):
    """A tiny addition policy: input is normalized (a, b), output is a distribution over answers 0..8."""

    def __init__(self, hidden=64):
        super().__init__()
        self.fc1 = torch.nn.Linear(2, hidden)
        self.fc2 = torch.nn.Linear(hidden, 9)

    def forward(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc2(h)

def sample_batch(n, seed):
    """Randomly generate n one-digit addition problems, a, b in 0..4, answers 0..8."""
    rng = np.random.RandomState(seed)
    a = rng.randint(0, 5, size=n)
    b = rng.randint(0, 5, size=n)
    x = torch.tensor(np.stack([a, b], axis=1), dtype=torch.float32) / 5.0
    y = torch.tensor(a + b, dtype=torch.long)
    return x, y

def grpo_update(policy, opt, x, y, G=32):
    """One GRPO-style update on a batch of problems (REINFORCE + within-group normalized advantage).

    x: [B, 2] normalized features; y: [B] true answers; G: number of outputs sampled per problem.
    Return (sampling accuracy, loss).
    """
    B = x.shape[0]
    xg = x.repeat_interleave(G, 0)
    yg = y.repeat_interleave(G, 0)
    logits = policy(xg)
    probs = torch.softmax(logits, dim=-1)
    actions = torch.multinomial(probs, 1).squeeze(1)
    log_probs = -F.cross_entropy(logits, actions, reduction="none")
    rewards = torch.where(actions == yg,
                          torch.ones_like(yg, dtype=torch.float32),
                          -torch.ones_like(yg, dtype=torch.float32))
    r_grp = rewards.view(B, G)
    mean_r = r_grp.mean(dim=1, keepdim=True)
    std_r = r_grp.std(dim=1, keepdim=True, unbiased=False) + 1e-8  # all-correct / all-wrong groups: adv≈0
    adv = ((r_grp - mean_r) / std_r).reshape(-1)
    loss = -(log_probs * adv).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    return (actions == yg).float().mean().item(), loss.item()

policy = ToyPolicy()
opt = torch.optim.Adam(policy.parameters(), lr=0.05)
x_train, y_train = sample_batch(25, seed=3)   # all 25 problems in 0..4
x_eval, y_eval = sample_batch(25, seed=9)

train_acc, eval_acc, entropies = [], [], []
for epoch in range(80):
    acc, loss = grpo_update(policy, opt, x_train, y_train)
    with torch.no_grad():
        pred = policy(x_eval).argmax(dim=-1)
        eval_acc.append((pred == y_eval).float().mean().item())
        p = torch.softmax(policy(x_eval), dim=-1)
        entropies.append((-p * torch.log(p + 1e-8)).sum(-1).mean().item())
    train_acc.append(acc)
    if epoch % 10 == 0:
        print(f"epoch {epoch:3d}: sample accuracy {acc:.3f}, eval accuracy {eval_acc[-1]:.3f}, "
              f"entropy {entropies[-1]:.3f}")

print("Key observation: within-group normalization makes correct outputs have positive advantage and incorrect ones negative,")
print("the policy concentrates probability on the correct answer, entropy falls, and accuracy rises from chance level to above 0.8.")


In [ ]:
# Training curves: accuracy rises, entropy falls
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(range(1, 81), train_acc, label="sample accuracy")
axes[0].plot(range(1, 81), eval_acc, label="eval accuracy")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("accuracy")
axes[0].set_title("GRPO improves the toy policy")
axes[0].legend()
axes[1].plot(range(1, 81), entropies)
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("mean entropy")
axes[1].set_title("Policy distribution concentrates")
plt.tight_layout()
plt.show()


## Summary

What this lecture covered:

- [ ] Test-time scaling changes the sampling distribution, train-time scaling changes model parameters; the former can only amplify existing ability, the latter changes ability itself
- [ ] STaR bootstrap loop: generate a rationale → filter by whether the answer is correct → rationalize failed samples → merge the dataset and fine-tune → generate again
- [ ] STaR is a policy-gradient approximation under an indicator reward; filtering is equivalent to dropping the gradient of zero-reward samples
- [ ] PPO needs a value network the same size as the policy, plus GAE; GRPO drops the critic, and advantage uses within-group statistics (r−mean)/std
- [ ] GRPO places the KL penalty in the objective, and uses the unbiased estimator exp(−log_ratio) + log_ratio − 1 to keep it non-negative
- [ ] Outcome supervision gives the whole output the same advantage; process supervision normalizes by step then accumulates, so per-token advantage is finer
- [ ] The unified paradigm puts SFT/RFT/Online RFT/GRPO in one framework; the differences are only data source, reward, and gradient coefficient
- [ ] DAPO's four engineering tricks: Clip-Higher preserves exploration, Dynamic Sampling filters all-correct / all-wrong groups, token-level loss treats tokens alike, overlong reward shaping suppresses truncation noise

## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.


**Exercise 1: GRPO within-group advantage**

Given the rewards of a group of 4 outputs `np.array([1.0, -1.0, 0.0, 1.0])` (two correct, one wrong, one partial), implement within-group normalized advantage `(r - mean) / std`.

Hint: compute the group mean first, then subtract the mean and divide by the population standard deviation; `np.std` defaults to `ddof=0`, matching the paper. An all-correct group has standard deviation 0, so the implementation needs to avoid 0/0.

In [ ]:
import numpy as np

def group_advantage(r):
    """Return within-group normalized advantage (r - mean) / std; if std=0 set 0."""
    mean = r.mean()                  # fill in
    std = r.std(ddof=0)              # fill in
    with np.errstate(divide="ignore", invalid="ignore"):
        adv = (r - mean) / std
    return np.nan_to_num(adv, nan=0.0, posinf=0.0, neginf=0.0)

adv = group_advantage(np.array([1.0, -1.0, 0.0, 1.0]))
expected = np.array([0.9045, -1.5076, -0.3015, 0.9045])
assert np.allclose(adv, expected, atol=1e-3), adv
assert np.allclose(group_advantage(np.ones(4)), 0.0)
print("Passed: an all-correct group's advantage is 0, so the gradient is 0.", np.round(adv, 4))


**Exercise 2: clipped surrogate objective**

Implement the PPO/GRPO clipped objective `min(ratio * A, clip(ratio, 1-eps, 1+eps) * A)`.

Hint: write `np.clip` and `np.minimum` separately: first compute the clipped ratio, then take the minimum with the unclipped result. Check three cases: A>0 caps at the upper bound, A<0 caps at the lower bound, and no clip inside the interval.

In [ ]:
import numpy as np

def clipped_surrogate(ratio, advantage, eps=0.2):
    """PPO/GRPO clipped objective min(ratio*A, clip(ratio, 1-eps, 1+eps)*A)."""
    clip_ratio = np.clip(ratio, 1 - eps, 1 + eps)                # fill in
    return np.minimum(ratio * advantage, clip_ratio * advantage)  # fill in

eps = 0.2
assert np.isclose(clipped_surrogate(1.8, 1.0, eps), 1.2)      # A>0 caps at the upper bound
assert np.isclose(clipped_surrogate(0.5, -1.0, eps), -0.8)    # A<0 caps at the lower bound
assert np.isclose(clipped_surrogate(1.1, 0.5, eps), 0.55)     # no clip inside the interval
print("Passed: when A>0 the ratio cannot overshoot; when A<0 probability cannot collapse.")


**Exercise 3: STaR filtering and rationalization data construction**

Given 5 generation records (each with rationale, answer, y) and 1 rationalization record, implement `filter_correct` and `rationalize_pool` to construct D_n and D_rat.

Hint: D_n keeps only samples that were generated correctly; D_rat keeps only samples that were generated wrong but rationalized correctly; when a rationalized sample enters the dataset, set `used_hint` to False, because at training time we pretend the model thought of it itself.

In [ ]:
records = [
    {"rationale": "add the ones place then carry", "answer": 12, "y": 12},
    {"rationale": "add the ones place then carry", "answer": 15, "y": 13},
    {"rationale": "split into two one-digit adds", "answer": 21, "y": 21},
    {"rationale": "add directly", "answer": 7, "y": 7},
    {"rationale": "estimate 30", "answer": 28, "y": 31},
]
rationalized = [
    {"rationale": "from the hint answer 31, recover the ones place and the carry", "answer": 31, "y": 31,
     "used_hint": True},
]

def filter_correct(records):
    """Return samples that were generated correctly."""
    return [r for r in records if r["answer"] == r["y"]]     # fill in

def rationalize_pool(records, rationalized):
    """Merge D_n and D_rat; rationalized samples contain no hint at training time."""
    D_n = filter_correct(records)
    D_rat = [{"rationale": r["rationale"], "answer": r["answer"],
              "y": r["y"], "used_hint": False} for r in rationalized]   # fill in
    return D_n, D_rat

D_n, D_rat = rationalize_pool(records, rationalized)
assert len(D_n) == 3
assert len(D_rat) == 1
assert all(not s["used_hint"] for s in D_rat)
assert all(s["answer"] == s["y"] for s in D_n + D_rat)
print("Passed: correct samples go into D_n, failed samples are rationalized into D_rat, and training pretends no hint was given.")


## References

- Zelikman et al., STaR: Bootstrapping Reasoning With Reasoning, arXiv:2203.14465 — original paper on the bootstrap-reasoning loop and rationalization
- Shao et al., DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models, arXiv:2402.03300 — source of the GRPO algorithm and the unified paradigm; advantage and loss formulas (Eq.1, Eq.3) come from this paper
- Yu et al., DAPO: An Open-Source LLM Reinforcement Learning System at Scale, arXiv:2503.14476 — four large-scale RL engineering tricks and the verl open-source system
- Schulman et al., Proximal Policy Optimization Algorithms, arXiv:1707.06347 — original source of the clipped objective and trust region; the reference for GRPO
- Schulman et al., High-Dimensional Continuous Control Using Generalized Advantage Estimation, arXiv:1506.02438 — definition of GAE, the source of advantage in PPO
- Wei et al., Chain-of-Thought Prompting Elicits Reasoning in Large Language Models, arXiv:2201.11903 — few-shot CoT prompting, the starting point of STaR
- DeepSeek-AI, DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning, arXiv:2501.12948 — the baseline DAPO is compared against
- Anthony et al., Expert Iteration, arXiv:1705.08439 — expert iteration, a theoretical relative of STaR
- verl framework, github.com/volcengine/verl — the RL training framework open-sourced with DAPO
- CS329A course syllabus, cs329a.stanford.edu — where this lecture sits in the course